### 1. 환경 설정

In [15]:
import subprocess, os

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

if IN_COLAB:
    subprocess.run(["apt-get", "install", "-y", "openslide-tools", "libgl1"],
                   capture_output=True)
else:
    # 로컬 실행 시 openslide 시스템 패키지를 미리 설치하세요.
    #   macOS : brew install openslide
    #   Ubuntu: sudo apt-get install -y openslide-tools libgl1
    pass

subprocess.run([
    "pip", "install", "-q",
    "openslide-python==4.0.0", "timm", "huggingface_hub",
    "h5py", "einops", "tqdm", "requests"
])

# CLAM 클론 (Colab에서만 필요 — 로컬에서는 이 노트북이 이미 CLAM 저장소 안에 있다고 가정)
if IN_COLAB and not os.path.exists('/content/CLAM'):
    subprocess.run(["git", "clone",
                    "https://github.com/mahmoodlab/CLAM.git",
                    "/content/CLAM"])

# 작업 디렉토리 (Colab: 임시 디스크 — 세션 종료 시 자동 삭제 / 로컬: TNBC_WORK_DIR 환경변수로 변경 가능)
RAW_SLIDE_DIR = '/content/raw_slides' if IN_COLAB else os.environ.get(
    'TNBC_WORK_DIR', os.path.join(os.getcwd(), 'raw_slides'))
os.makedirs(RAW_SLIDE_DIR, exist_ok=True)

# 특징 파일 저장 디렉토리 (Colab: Drive 영구 보관 / 로컬: TNBC_PROJECT_ROOT 환경변수로 변경 가능)
DRIVE_ROOT        = ('/content/drive/MyDrive/TCGA_BRCA_project' if IN_COLAB
                      else os.environ.get('TNBC_PROJECT_ROOT', os.path.expanduser('~/TCGA_BRCA_project')))
DRIVE_FEATURE_DIR = f'{DRIVE_ROOT}/features'
DRIVE_LOG_DIR     = f'{DRIVE_ROOT}/logs'
os.makedirs(DRIVE_FEATURE_DIR, exist_ok=True)
os.makedirs(DRIVE_LOG_DIR,     exist_ok=True)

print(f"✓ 환경 설정 완료 ({'Colab' if IN_COLAB else 'Local'})")
print(f"  feature 파일 저장 위치: {DRIVE_FEATURE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ 환경 설정 완료
  feature 파일 저장 위치: /content/drive/MyDrive/TCGA_BRCA_project/features


In [2]:

# =============================================================================
# 셀 2: 임상 라벨 수집 (cBioPortal) — 캐싱 적용
# =============================================================================

import requests, pandas as pd

clinical_path = f'{DRIVE_LOG_DIR}/clinical_labels.csv'

if os.path.exists(clinical_path):
    # Drive에 저장된 파일이 있으면 API 재호출 없이 바로 로드
    print("기존 clinical_labels.csv 발견 — API 재호출 없이 불러옵니다.")
    clinical_df = pd.read_csv(clinical_path)
    print(f"  환자 수: {len(clinical_df)}명")
    print(f"  라벨 분포:\n{clinical_df['label_str'].value_counts().to_string()}")

else:
    # 최초 1회만 API 호출
    def get_clinical_labels_cbioportal(study_id="brca_tcga"):
        print(f"cBioPortal에서 임상 데이터 조회 중 (study: {study_id})...")

        url = f"https://www.cbioportal.org/api/studies/{study_id}/clinical-data"
        params = {"clinicalDataType": "PATIENT", "projection": "DETAILED"}
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        df_raw = pd.DataFrame(r.json())

        attrs_needed = ['ER_STATUS_BY_IHC', 'PR_STATUS_BY_IHC', 'IHC_HER2']
        df_wide = (
            df_raw[df_raw['clinicalAttributeId'].isin(attrs_needed)]
            .pivot_table(index='patientId', columns='clinicalAttributeId',
                         values='value', aggfunc='first')
            .reset_index()
        )
        df_wide.columns.name = None
        # patientId → patient_id로 컬럼명 통일
        df_wide = df_wide.rename(columns={'patientId': 'patient_id'})

        def assign_label(row):
            er   = str(row.get('ER_STATUS_BY_IHC', '')).strip()
            pr   = str(row.get('PR_STATUS_BY_IHC', '')).strip()
            her2 = str(row.get('IHC_HER2', '')).strip()
            if er == 'Negative' and pr == 'Negative' and her2 == 'Negative':
                return 1, 'TNBC'
            elif 'Positive' in [er, pr, her2]:
                return 0, 'non-TNBC'
            else:
                return -1, 'Unknown'  # Indeterminate, Equivocal → 제외

        df_wide[['label', 'label_str']] = df_wide.apply(
            lambda r: pd.Series(assign_label(r)), axis=1
        )
        df_clean = df_wide[df_wide['label'] != -1].reset_index(drop=True)

        print(f"  전체 환자: {len(df_wide)}명")
        print(f"  라벨 확정: {len(df_clean)}명")
        print(f"  라벨 분포:\n{df_clean['label_str'].value_counts().to_string()}")
        return df_clean

    clinical_df = get_clinical_labels_cbioportal()
    clinical_df.to_csv(clinical_path, index=False)
    print(f"\n✓ 임상 라벨 저장 완료")


기존 clinical_labels.csv 발견 — API 재호출 없이 불러옵니다.
  환자 수: 979명
  라벨 분포:
label_str
non-TNBC    863
TNBC        116


In [3]:

# =============================================================================
# 셀 3: GDC open access 슬라이드 목록 조회 + 임상 라벨 매칭 — 캐싱 적용
# =============================================================================

import json

manifest_path = f'{DRIVE_LOG_DIR}/manifest.csv'

if os.path.exists(manifest_path):
    # Drive에 저장된 파일이 있으면 API 재호출 없이 바로 로드
    print("기존 manifest.csv 발견 — API 재호출 없이 불러옵니다.")
    manifest = pd.read_csv(manifest_path)
    print(f"  불러온 환자 수: {len(manifest)}명")
    print(f"  라벨 분포:\n{manifest['label_str'].value_counts().to_string()}")

else:
    # 최초 1회만 API 호출
    def get_open_access_slides(n=3112):
        """
        GDC에서 TCGA-BRCA open access 슬라이드 목록 조회.
        토큰 불필요. 총 3,112개 확인됨.
        환자당 여러 슬라이드가 있을 수 있으므로 첫 번째 슬라이드만 유지.
        """
        print(f"GDC open access 슬라이드 조회 중 (n={n})...")
        endpoint = "https://api.gdc.cancer.gov/files"
        filters = {
            "op": "and",
            "content": [
                {"op": "=", "content": {"field": "cases.project.project_id",
                                         "value": "TCGA-BRCA"}},
                {"op": "=", "content": {"field": "data_type",
                                         "value": "Slide Image"}},
                {"op": "=", "content": {"field": "access",
                                         "value": "open"}}
            ]
        }
        params = {
            "filters": json.dumps(filters),
            "fields": "file_id,file_name,cases.submitter_id",
            "size": str(n),
            "format": "JSON"
        }
        r = requests.get(endpoint, params=params)
        r.raise_for_status()
        hits = r.json()['data']['hits']

        records = []
        for h in hits:
            case = h['cases'][0] if h.get('cases') else {}
            records.append({
                'file_id':    h['file_id'],
                'file_name':  h['file_name'],
                'patient_id': case.get('submitter_id', '')
            })

        df = pd.DataFrame(records)
        df_dedup = df.drop_duplicates(subset='patient_id', keep='first')
        print(f"  전체 슬라이드: {len(df)}개 → 고유 환자: {len(df_dedup)}명")
        return df_dedup

    def download_open_slide(file_id, save_path):
        """토큰 없이 GDC open access 파일 다운로드"""
        url = f"https://api.gdc.cancer.gov/data/{file_id}"
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(save_path, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    slide_df = get_open_access_slides()
    manifest = slide_df.merge(clinical_df, on='patient_id', how='inner').reset_index(drop=True)

    print(f"\n매칭 결과:")
    print(f"  최종: {len(manifest)}명")
    print(f"  라벨 분포:\n{manifest['label_str'].value_counts().to_string()}")

    manifest.to_csv(manifest_path, index=False)
    print(f"\n✓ 매니페스트 저장 완료")

# download_open_slide는 셀 5에서 사용하므로 항상 정의
def download_open_slide(file_id, save_path):
    """토큰 없이 GDC open access 파일 다운로드"""
    url = f"https://api.gdc.cancer.gov/data/{file_id}"
    with requests.get(url, stream=True, timeout=600) as r:
        r.raise_for_status()
        with open(save_path, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)


기존 manifest.csv 발견 — API 재호출 없이 불러옵니다.
  불러온 환자 수: 979명
  라벨 분포:
label_str
non-TNBC    863
TNBC        116


In [5]:

# =============================================================================
# 셀 4: UNI 모델 로드
# =============================================================================

import torch
import timm
from huggingface_hub import login
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform

# HuggingFace 토큰 (MahmoodLab/UNI 접근 승인 후 발급)
# https://huggingface.co/settings/tokens
# 토큰 값은 코드에 직접 넣지 않습니다. Colab Secrets(좌측 열쇠 아이콘)에 'HF_TOKEN'으로 등록해두면 자동으로 읽어옵니다.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    HF_TOKEN = getpass('Hugging Face 토큰을 입력하세요: ')
login(token=HF_TOKEN, add_to_git_credential=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

print("\nUNI 모델 로드 중 (최초 실행 시 약 2분 소요)...")
model = timm.create_model(
    "hf-hub:MahmoodLab/UNI",
    pretrained=True,
    init_values=1e-5,
    num_classes=0   # 분류 헤드 제거 → 순수 임베딩 출력
)
model = model.to(device)
model.eval()

# UNI 공식 전처리 transform
transform = create_transform(
    **resolve_data_config(model.pretrained_cfg, model=model)
)

print(f"✓ UNI 로드 완료")
print(f"  출력 임베딩 차원: {model.num_features}")  # 1024


사용 디바이스: cpu

UNI 모델 로드 중 (최초 실행 시 약 2분 소요)...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.21GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

✓ UNI 로드 완료
  출력 임베딩 차원: 1024


In [11]:
# !pip uninstall openslide-python -y
# !apt-get remove -y python3-openslide
# !apt-get install -y openslide-tools
# !pip install openslide-python==1.4.6

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'python3-openslide' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
openslide-tools is already the newest version (3.4.1+dfsg-5build1).
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.


In [12]:
# =============================================================================
# 셀 5: 처리 함수 정의
# =============================================================================

import openslide
import numpy as np
import time

PATCH_SIZE = 256   # 패치 크기 (픽셀)
BATCH_SIZE = 32    # UNI 배치 크기
BG_THRESH  = 230   # 배경 필터 — 평균 픽셀값 상한 (클수록 밝음=배경)
BG_STD_MIN = 10    # 배경 필터 — 표준편차 하한 (너무 단색이면 배경)

def calculate_max_patches(slide_path, patch_size=PATCH_SIZE,
                           sample_ratio=0.3, hard_max=8000, hard_min=1000):
    """
    슬라이드 실제 픽셀 크기를 기반으로 max_patches를 동적으로 계산.
    큰 슬라이드는 많이, 작은 슬라이드는 적게 뽑아 커버리지를 균등하게 유지.

    sample_ratio : 추정 조직 패치 수 대비 샘플링 비율
    hard_max     : 메모리/시간 제한을 위한 상한선
    hard_min     : 너무 작은 슬라이드를 위한 하한선
    """
    slide = openslide.OpenSlide(slide_path)
    w, h  = slide.level_dimensions[0]
    slide.close()

    total_grid       = (w // patch_size) * (h // patch_size)
    estimated_tissue = int(total_grid * 0.4)  # 조직 비율 약 40% 가정
    max_patches      = int(estimated_tissue * sample_ratio)
    max_patches      = max(hard_min, min(max_patches, hard_max))

    print(f"  슬라이드 실제 크기: {w:,} × {h:,} 픽셀")
    print(f"  처리 대상 크기    : {(w//patch_size)*patch_size:,} × "
          f"{(h//patch_size)*patch_size:,} 픽셀 "
          f"({(w//patch_size)*patch_size*100//w}% × "
          f"{(h//patch_size)*patch_size*100//h}% 커버)")
    print(f"  추정 조직 패치    : {estimated_tissue:,}개")
    print(f"  max_patches 설정  : {max_patches:,}개")

    return max_patches, w, h


def extract_features_streaming(slide_path, patch_size=PATCH_SIZE,
                                batch_size=BATCH_SIZE, max_patches=5000):
    """
    스트리밍 방식 패치 추출 + UNI 임베딩.
    패치를 전부 메모리에 올리지 않고 배치 단위로 즉시 임베딩하여
    메모리 사용량을 항상 배치 크기(32개)분으로 유지.

    반환:
        features : torch.Tensor (n_patches, 1024) 또는 None
        coords   : list of (x, y) 튜플
    """
    slide         = openslide.OpenSlide(slide_path)
    w, h          = slide.level_dimensions[0]
    all_features  = []
    all_coords    = []
    batch_patches = []
    batch_coords  = []
    total_patches = 0

    for y in range(0, h - patch_size, patch_size):
        for x in range(0, w - patch_size, patch_size):
            if total_patches >= max_patches:
                break

            patch     = slide.read_region((x, y), 0, (patch_size, patch_size))
            patch_rgb = patch.convert('RGB')
            arr       = np.array(patch_rgb)

            # 배경 필터: 너무 밝거나 단색이면 건너뜀
            if arr.mean() >= BG_THRESH or arr.std() <= BG_STD_MIN:
                continue

            batch_patches.append(patch_rgb)
            batch_coords.append((x, y))
            total_patches += 1

            # 배치가 차면 즉시 임베딩 후 메모리 해제
            if len(batch_patches) == batch_size:
                with torch.no_grad():
                    tensors = torch.stack(
                        [transform(p) for p in batch_patches]
                    ).to(device)
                    feats = model(tensors).cpu()
                all_features.append(feats)
                all_coords.extend(batch_coords)
                del batch_patches, batch_coords, tensors
                batch_patches = []
                batch_coords  = []

        if total_patches >= max_patches:
            break

    # 루프 종료 후 남은 배치 처리
    if batch_patches:
        with torch.no_grad():
            tensors = torch.stack(
                [transform(p) for p in batch_patches]
            ).to(device)
            feats = model(tensors).cpu()
        all_features.append(feats)
        all_coords.extend(batch_coords)
        del batch_patches, tensors

    slide.close()

    if not all_features:
        return None, []

    return torch.cat(all_features, dim=0), all_coords


def process_one_patient(row, slide_tmp_dir='/content/raw_slides'):
    """
    환자 1명 전체 처리 파이프라인:
    다운로드 → 슬라이드 크기 확인 → 동적 max_patches 계산
    → 스트리밍 패치 추출 + UNI 임베딩 → Drive 저장 → 원본 삭제
    """
    patient_id = row['patient_id']
    file_id    = row['file_id']
    file_name  = row['file_name']
    label      = row['label']
    label_str  = row['label_str']

    save_path = os.path.join(DRIVE_FEATURE_DIR, f"{patient_id}.pt")

    # 이미 처리된 환자 건너뜀 → 세션 재시작 후 이어하기 가능
    if os.path.exists(save_path):
        print(f"  [{patient_id}] 이미 처리됨, 건너뜀")
        return {'patient_id': patient_id, 'status': 'skipped',
                'label': label, 'label_str': label_str}

    result   = {'patient_id': patient_id, 'label': label, 'label_str': label_str}
    svs_path = os.path.join(slide_tmp_dir, file_name)

    try:
        # 1. 다운로드
        t0 = time.time()
        print(f"  다운로드 중...")
        download_open_slide(file_id, svs_path)
        size_gb  = os.path.getsize(svs_path) / 1e9
        dl_time  = time.time() - t0
        print(f"  다운로드 완료 ({size_gb:.2f} GB, {dl_time:.0f}초)")

        # 2. 슬라이드 크기 확인 + 동적 max_patches 계산
        max_patches, slide_w, slide_h = calculate_max_patches(svs_path)

        # 3. 스트리밍 패치 추출 + UNI 임베딩
        print(f"  패치 추출 + 임베딩 중 (스트리밍)...")
        t1 = time.time()
        features, coords = extract_features_streaming(
            svs_path, max_patches=max_patches
        )
        proc_time = time.time() - t1

        if features is None:
            raise ValueError("조직 패치 없음")

        # 실제 커버된 픽셀 범위 계산
        if coords:
            max_x = max(c[0] for c in coords) + PATCH_SIZE
            max_y = max(c[1] for c in coords) + PATCH_SIZE
        else:
            max_x = max_y = 0

        print(f"  패치 수           : {len(coords):,}개")
        print(f"  임베딩 shape      : {tuple(features.shape)}")
        print(f"  실제 커버 영역    : {max_x:,} × {max_y:,} 픽셀 "
              f"({max_x*100//slide_w}% × {max_y*100//slide_h}% 커버)")
        print(f"  추출+임베딩 시간  : {proc_time:.0f}초")

        # 4. Drive 저장
        torch.save({
            'features':    features,              # (n_patches, 1024)
            'coords':      torch.tensor(coords),  # (n_patches, 2)
            'patient_id':  patient_id,
            'label':       label,
            'label_str':   label_str,
            'n_patches':   len(coords),
            'slide_size':  (slide_w, slide_h),
            'max_patches': max_patches
        }, save_path)
        pt_size_mb = os.path.getsize(save_path) / 1e6
        print(f"  Drive 저장 완료   : {pt_size_mb:.1f} MB")

        result.update({
            'status':      'success',
            'n_patches':   len(coords),
            'size_gb':     round(size_gb, 2),
            'slide_w':     slide_w,
            'slide_h':     slide_h,
            'max_patches': max_patches,
            'pt_size_mb':  round(pt_size_mb, 1),
            'dl_time_s':   round(dl_time),
            'proc_time_s': round(proc_time)
        })

    except Exception as e:
        print(f"  ✗ 오류: {e}")
        result['status'] = f'error: {str(e)}'

    finally:
        # 5. 원본 삭제 (성공/실패 무관)
        if os.path.exists(svs_path):
            os.remove(svs_path)
            print(f"  원본 삭제 완료")

    return result


✓ 셀 5 함수 정의 완료


In [ ]:
# =============================================================================
# 셀 6: 전체 배치 처리 실행
# =============================================================================

import shutil
from sklearn.model_selection import train_test_split

def check_drive_space(warn_gb=10):
    total, used, free = shutil.disk_usage('/content/drive/MyDrive')
    free_gb = free / 1e9
    print(f"Drive 용량: 전체 {total/1e9:.1f}GB / 사용 {used/1e9:.1f}GB / 여유 {free_gb:.1f}GB")
    if free_gb < warn_gb:
        print(f"⚠ Drive 여유 공간 {warn_gb}GB 미만 — 처리를 중단하세요.")
        return False
    return True


def select_balanced_subset(manifest, n_per_class=116, random_state=42):
    """
    TNBC 전체(116명) + non-TNBC 층화 랜덤 샘플링(116명).
    층화 기준: ER/PR/HER2 조합 → non-TNBC 내 서브타입 비율 유지.
    random_state=42 고정 → 세션 재시작해도 항상 동일한 330명 선택.
    """
    tnbc    = manifest[manifest['label_str'] == 'TNBC'].copy()
    nontnbc = manifest[manifest['label_str'] == 'non-TNBC'].copy()

    nontnbc['subtype_group'] = (
        nontnbc['ER_STATUS_BY_IHC'] + '_' +
        nontnbc['PR_STATUS_BY_IHC'] + '_' +
        nontnbc['IHC_HER2']
    )

    nontnbc_sampled, _ = train_test_split(
        nontnbc,
        train_size=min(n_per_class, len(nontnbc)),
        stratify=nontnbc['subtype_group'],
        random_state=random_state
    )

    subset = pd.concat([tnbc, nontnbc_sampled]).reset_index(drop=True)
    print(f"처리 대상 선택 완료:")
    print(f"  TNBC     : {len(tnbc)}명 (전체 사용)")
    print(f"  non-TNBC : {len(nontnbc_sampled)}명 (층화 샘플링)")
    print(f"  합계     : {len(subset)}명")
    return subset


# ── 처리 대상 선택 ────────────────────────────────────────────────────────────
manifest_subset = select_balanced_subset(manifest, n_per_class=116)

# 다계정 병렬 처리 시: 아래처럼 구간을 나눠서 실행
# 계정 A: manifest_subset = manifest_subset.iloc[0:110]
# 계정 B: manifest_subset = manifest_subset.iloc[110:220]
# 계정 C: manifest_subset = manifest_subset.iloc[220:]

# ── 기존 로그 불러오기 (이어하기) ────────────────────────────────────────────
log_path = f'{DRIVE_LOG_DIR}/processing_log.csv'
# 다계정 병렬 처리 시에는 계정별로 다른 로그 파일 사용
# log_path = f'{DRIVE_LOG_DIR}/processing_log_A.csv'

log_rows = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []

print(f"\n처리 대상: {len(manifest_subset)}명")
print(f"라벨 분포:\n{manifest_subset['label_str'].value_counts().to_string()}\n")
check_drive_space()
print("=" * 60)

# ── 메인 처리 루프 ────────────────────────────────────────────────────────────
total = len(manifest_subset)
for idx, row in manifest_subset.iterrows():
    print(f"\n[{idx+1}/{total}] {row['patient_id']} ({row['label_str']})")
    result = process_one_patient(row)
    log_rows.append(result)

    # 매 처리 후 로그 저장 (세션 끊겨도 진행 상황 보존)
    pd.DataFrame(log_rows).to_csv(log_path, index=False)

    # 10명마다 Drive 용량 체크
    if (idx + 1) % 10 == 0:
        print()
        if not check_drive_space():
            print("처리 중단.")
            break

# ── 최종 요약 ─────────────────────────────────────────────────────────────────
log_df  = pd.DataFrame(log_rows)
success = log_df[log_df['status'] == 'success']
skipped = log_df[log_df['status'] == 'skipped']
errors  = log_df[log_df['status'].str.startswith('error', na=False)]

print(f"\n{'='*60}")
print("처리 완료 요약")
print(f"{'='*60}")
print(f"  성공: {len(success)}명 / 건너뜀: {len(skipped)}명 / 오류: {len(errors)}명")
if len(success) > 0:
    print(f"\n  패치 수 통계:")
    print(f"    평균: {success['n_patches'].mean():,.0f}  "
          f"최소: {success['n_patches'].min():,}  "
          f"최대: {success['n_patches'].max():,}")
if len(errors) > 0:
    print(f"\n  오류 목록:")
    print(errors[['patient_id', 'status']].to_string())

In [18]:
# 셀 7: 특징 파일 검증 및 CLAM 입력 형식 준비 (수정본)

def verify_and_prepare_dataset(feature_dir, log_dir):
    print("특징 파일 검증 중...")

    records = []
    feature_files = [f for f in os.listdir(feature_dir) if f.endswith('.pt')]

    for fname in feature_files:
        fpath = os.path.join(feature_dir, fname)
        try:
            data = torch.load(fpath, map_location='cpu')
            records.append({
                'slide_id':    data['patient_id'],
                'label':       data['label'],
                'label_str':   data['label_str'],
                'n_patches':   data['n_patches'],
                'slide_w':     data['slide_size'][0],
                'slide_h':     data['slide_size'][1],
                'max_patches': data['max_patches'],
                'feat_shape':  str(tuple(data['features'].shape)),
                'file_path':   fpath,
                'status':      'ok'
            })
        except Exception as e:
            records.append({
                'slide_id': fname.replace('.pt', ''),
                'file_path': fpath,
                'status': f'corrupt: {e}'
            })

    df = pd.DataFrame(records)
    ok_df  = df[df['status'] == 'ok'].reset_index(drop=True)
    bad_df = df[df['status'] != 'ok']

    print(f"\n검증 결과:")
    print(f"  정상 파일: {len(ok_df)}개")
    print(f"  손상 파일: {len(bad_df)}개")

    if len(ok_df) > 0:
        print(f"\n라벨 분포:")
        print(ok_df['label_str'].value_counts().to_string())
        print(f"\n패치 수 통계:")
        print(f"  평균: {ok_df['n_patches'].mean():,.0f}  "
              f"최소: {ok_df['n_patches'].min():,}  "
              f"최대: {ok_df['n_patches'].max():,}")
        print(f"\n슬라이드 크기 통계 (픽셀):")
        print(f"  W 평균: {ok_df['slide_w'].mean():,.0f}  "
              f"최소: {ok_df['slide_w'].min():,}  "
              f"최대: {ok_df['slide_w'].max():,}")
        print(f"  H 평균: {ok_df['slide_h'].mean():,.0f}  "
              f"최소: {ok_df['slide_h'].min():,}  "
              f"최대: {ok_df['slide_h'].max():,}")

    if len(bad_df) > 0:
        print(f"\n손상 파일 목록:")
        print(bad_df[['slide_id', 'status']].to_string())

    # CLAM 학습용 CSV 생성
    # 형식: slide_id, label, label_str, file_path
    clam_csv_path = os.path.join(log_dir, 'dataset_for_clam.csv')
    ok_df[['slide_id', 'label', 'label_str', 'file_path']].to_csv(
        clam_csv_path, index=False
    )
    print(f"\n✓ CLAM 학습용 CSV 저장: {clam_csv_path}")

    return ok_df


dataset_df = verify_and_prepare_dataset(DRIVE_FEATURE_DIR, DRIVE_LOG_DIR)
print("\n모든 단계 완료. 다음 단계: CLAM 학습")

특징 파일 검증 중...

검증 결과:
  정상 파일: 330개
  손상 파일: 0개

라벨 분포:
label_str
non-TNBC    214
TNBC        116

패치 수 통계:
  평균: 5,577  최소: 199  최대: 8,000

슬라이드 크기 통계 (픽셀):
  W 평균: 95,044  최소: 8,618  최대: 197,263
  H 평균: 45,112  최소: 7,243  최대: 247,552

✓ CLAM 학습용 CSV 저장: /content/drive/MyDrive/TCGA_BRCA_project/logs/dataset_for_clam.csv

모든 단계 완료. 다음 단계: CLAM 학습
